In [ ]:
from aistudio_common.utils.notebook_utils import NotebookUtils
from aistudio_common.utils import env_utils
from aistudio_notebook.utils.notebook_extension_util import topai, get_targets, get_args

"""
notebook 默认初始化代码，导入常用工具包，定义df1、df2、df3、df4、o，若不使用可删除该cell
"""
o = env_utils.get_odps_instance()
args = get_args()
targets = get_targets()
d1 = NotebookUtils.get_df(1)
d2 = NotebookUtils.get_df(2)
df = d1.to_pandas()
df1 = d2.to_pandas()
print(len(df))
print(len(df1))

df_result = df[~df["supply_code"].isin(df1["benefit_id"])]
print(len(df_result))
print(df_result.head())
global_t = len(df_result)

bid1 = df1["benefit_id"].tolist()
bsid1 = df1["sid_v1"].tolist()

print(bid1[:10])
print(bsid1[:10])

In [ ]:
from pypai.utils import env_utils
import os
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm import tqdm  
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from vllm import LLM
from odps import DataFrame
from aistudio_modelhub.core.modelhub_ops import download
import torch
from rqvae import RQVAE

with open("./id.txt", 'w') as f:
    for item in df_result['supply_code']:
        f.write(item + "\n")

# 封装单个任务的处理逻辑
def process_single_row(row, output_dir, session):
    s_code = str(row['supply_code']).strip()
    s_name = str(row['supply_name']).strip()
    s_url = row['supply_pic']
    
    txt_path = os.path.join(output_dir, s_code + ".txt")
    img_path = os.path.join(output_dir, s_code + ".jpg")
    
    try:
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(s_name)
            
        if not s_url or pd.isna(s_url):
            raise ValueError("URL is empty")

        response = session.get(s_url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content)).convert("RGB")
        image.save(img_path, format="JPEG", quality=95)
        
        return True, None

    except Exception as e:
        
        if os.path.exists(txt_path):
            os.remove(txt_path)
        return False, s_code + ": " + str(e)

def build_dataset_concurrently(df, output_dir = "./multimodal_dataset_%s" % env_utils.get_bizdate(), max_workers = 16):
   
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    })

    success_count = 0
    fail_count = 0
    
    print("Start processing" +str(len(df)) +" records with " + str(max_workers)+ " threads...")

    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        futures = [executor.submit(process_single_row, row, output_dir, session) for _, row in df.iterrows()]
        
        
        for future in tqdm(as_completed(futures), total=len(futures)):
            is_success, error_msg = future.result()
            if is_success:
                success_count += 1
            else:
                fail_count += 1
               

    print("\nDone! Success:" + str(success_count) + ", Failed: "+ str(fail_count))

if global_t != 0:
    build_dataset_concurrently(df_result, max_workers=16)

In [ ]:
download(
    modelhub_id=31600008,
    modelhub_version_id=102300043,
    local_path="/home/admin/Qwen3-VL-Embedding-8B",
    enable_mount=True
)
print("zy model download successfully")

In [ ]:
BATCH_SIZE = 128  
model = LLM(
    model="/home/admin/Qwen3-VL-Embedding-8B", 
    runner="pooling", # embedding 模式需要 pooling
    gpu_memory_utilization=0.9,
    max_num_seqs=BATCH_SIZE, 
    enforce_eager=False 
)
print("zy model load successfully")

In [ ]:
def get_image_from_local(image_path: str):
    if not os.path.exists(image_path):
        return "error"
    try:
        img = Image.open(image_path).convert("RGB")
        img.load() 
        return img
    except Exception:
        return "error"

def get_title_from_local(item_id):
    path = "./multimodal_dataset_{}/{}.txt".format(env_utils.get_bizdate(), item_id)
    if not os.path.exists(path):
        return "error"
    try:
        with open(path, 'r', encoding='utf-8') as o:
            title = o.readlines()[0].strip("\n")
        return title
    except Exception:
        return "error"

def prepare_single_input(item_id):
    try:
        image_path = "./multimodal_dataset_{}/{}.jpg".format(env_utils.get_bizdate(), item_id)
        image = get_image_from_local(image_path)
        title = get_title_from_local(item_id)
        
        if image == 'error' or title == 'error':
            return None
            
        image_placeholder = "<|vision_start|><|image_pad|><|vision_end|>"
        sub = {
            "prompt": f"{image_placeholder}\n{title}",
            "multi_modal_data": {"image": image},
        }
        return sub
    except Exception:
        return None


if global_t != 0:
    if os.path.exists("./id.txt"):
        with open("./id.txt", 'r') as f:
            all_ids = [item.strip("\n") for item in f.readlines()]
    else:
        print("not found id.txt")
        all_ids = []

    total_items = len(all_ids)
    print(f"all num: {total_items}")

    with open("./member_feeds_embedding.txt", 'w', encoding='utf-8') as f_out:
        
        pbar = tqdm(total=total_items, desc="Inference Progress")
        
        for i in range(0, total_items, BATCH_SIZE):
            batch_ids_raw = all_ids[i : i + BATCH_SIZE]
            
            batch_inputs = []
            batch_valid_ids = []

            for item_id in batch_ids_raw:
                input_data = prepare_single_input(item_id)
                if input_data is not None:
                    batch_inputs.append(input_data)
                    batch_valid_ids.append(item_id)
            
            if not batch_inputs:
                pbar.update(len(batch_ids_raw))
                print("error")
                continue

            try:
                outputs = model.embed(batch_inputs)
      
                lines_to_write = []
              
                for j, output_item in enumerate(outputs):
                    current_id = batch_valid_ids[j]
                 
                    emb = output_item.outputs.embedding
                  
                    lines_to_write.append(f"{current_id}:{json.dumps(emb)}\n")
                
                f_out.writelines(lines_to_write)
                f_out.flush()
                print(lines_to_write[0])

            except Exception as e:
                print(f"Error in inference batch {i}: {e}")
                import traceback
                traceback.print_exc()
            
            # 更新进度条
            pbar.update(len(batch_ids_raw))

    pbar.close()

In [ ]:
if global_t != 0:
    device = torch.device('cuda')
    model = RQVAE()

    checkpoint = torch.load('./rqvae_model.pt', map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])

    model.eval() 
    print("model load GPU successfully！")

In [ ]:
if global_t != 0:
    vec_dict = {}
    with open("./member_feeds_embedding.txt", 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            parts = line.split(':', 1)
            if len(parts) != 2:
                continue

            id_str = parts[0]
            vec_str = parts[1].strip('[]')

            vec = [float(x.strip()) for x in vec_str.split(',')]
            vec_dict[id_str] = torch.tensor([vec], dtype=torch.float32)
            if i%500 == 0:
                print(i)
    print("embedding file write successfully!")

In [ ]:
if global_t != 0:
    res = {}
    i = 0
    for k,v in vec_dict.items():
        codes = model.get_codes(v)
        res[k] = [codes[0][0].item(), codes[0][1].item(), codes[0][2].item()]
        i += 1
        if i%500==0:
            print(i)

    with open("member_feeds_sid_delta.txt", 'w') as f:
        for k,v in res.items():
            f.write(k+":"+str(v)+"\n")
            print(k+":"+str(v))
    print("delta write successfully!")

In [ ]:
if global_t != 0:
    benefit_id = []
    sid_v1 = []

    with open("./member_feeds_sid_delta.txt", 'r') as f:
    for item in f.readlines():
        s = item.split(":")
        benefit_id.append(s[0])
        sid_v1.append(s[1].strip("\n").strip("[]"))

    benefit_id += bid1
    sid_v1 += bsid1

    data = pd.DataFrame()
    code1 = []
    code2 = []
    code3 = []
    for i in range(len(benefit_id)):
    code1.append(sid_v1[i].split(",")[0].strip(" "))
    code2.append(sid_v1[i].split(",")[1].strip(" "))
    code3.append(sid_v1[i].split(",")[2].strip(" "))
    data['benefit_id'] = benefit_id
    data['sid_v1'] = sid_v1
    data['code1'] = code1
    data['code2'] = code2
    data['code3'] = code3

    topai(1, DataFrame(data))